In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:\\Users\\erons\\saas-churn-prediction\\data\\raw\\churn-bigml-80.csv")

df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


From the EDA heatmap, we saw that `charges` are highly correlated with minutes, so we drop them.

In [4]:
df = df.drop(columns=[
    "Total day charge",
    "Total eve charge",
    "Total night charge",
    "Total intl charge"
])

Usage charges are derived directly from usage minutes and therefore proivde redundant information. These features are removed to avoid mutollinearity.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   object 
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   object 
 4   Voice mail plan         2666 non-null   object 
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total eve minutes       2666 non-null   float64
 9   Total eve calls         2666 non-null   int64  
 10  Total night minutes     2666 non-null   float64
 11  Total night calls       2666 non-null   int64  
 12  Total intl minutes      2666 non-null   float64
 13  Total intl calls        2666 non-null   int64  
 14  Customer service calls  2666 non-null   

In [8]:
#check for all the columns that are categorical
df.select_dtypes(include=["object"]).columns

Index(['State', 'International plan', 'Voice mail plan'], dtype='object')

In [16]:
#drop the "state" column
df = df.drop(columns=["State"])

### Removing Area Code

The area code variable represents a geographic identifier for each customer. Although stored as a numeric value, it functions as a categorical label rather than a meaningful numerical measurement.

Since the dataset contains only a few area code categories and there is no clear evidence that geographic region strongly influences churn in this dataset, the variable is removed to simplify the feature space.

In [25]:
df = df.drop(columns=["Area code"])

The `State` column was removed because it has high cardinality (51 unique values) and does not provide a strong enough signal to justify the added complexity. Encoding it would create many features and may increase overfitting, so it was dropped to simplify the model.

## Convert Categorical Variables
Some columns are text data and need to be converted to numeric values to allow modelling

In [14]:
df["International plan"] = df["International plan"].map({"No":0, "Yes":1})
df["Voice mail plan"] = df["Voice mail plan"].map({"No":0, "Yes":1})

## Creating Behavioral Features

The original dataset contains several raw usage metrics such as minutes and call counts across different times of the day. While these variables are informative individually, they may not fully capture overall customer behavior.

Feature engineering is used to create new variables that better represent customer activity patterns. These engineered features combine multiple related variables to produce more meaningful indicators of engagement and service usage.

For example:

- **Total usage minutes** combines daytime, evening, night, and international usage to represent overall service usage.
- **Total calls** aggregates call counts across all time periods to measure overall communication activity.
- **Service call ratio** measures how frequently a customer contacts customer support relative to their total call activity.

These features aim to capture behavioral signals that may be associated with customer dissatisfaction or disengagement, which could increase the likelihood of churn.

### Feature 1 - Total Usage

In [18]:
df["Total usage minutes"] = (
    df["Total day minutes"] +
    df["Total eve minutes"] +
    df["Total night minutes"] +
    df["Total intl minutes"]
)

### Feature 2 - Total Calls

In [20]:
df["Total calls"] = (
    df["Total day calls"] +
    df["Total eve calls"] +
    df["Total night calls"] +
    df["Total intl calls"]
)

### Feature 3 - Service Pressure
This shows how often a customer's calls are related to support issues.

In [22]:
df["Service call ratio"] = df["Customer service calls"] / df["Total calls"]

Check correlation of new variables with `churn`

In [28]:
df.corr()["Churn"].sort_values(ascending=False)

Churn                     1.000000
International plan        0.277489
Customer service calls    0.202590
Service call ratio        0.196630
Total day minutes         0.195688
Total usage minutes       0.180345
Total intl minutes        0.086204
Total eve minutes         0.072906
Total night minutes       0.033639
Total day calls           0.018290
Account length            0.017728
Total night calls         0.012262
Total calls               0.011650
Total eve calls          -0.001539
Total intl calls         -0.069882
Number vmail messages    -0.086474
Voice mail plan          -0.099291
Name: Churn, dtype: float64

Save Processed Data

In [29]:
df.to_csv("../data/processed/churn_features.csv", index=False)

## Feature Engineering Summary

The following feature engineering steps were performed:

- Removed redundant charge features
- Removed location(`State` and `Area code`) columns
- Converted categorical variables to numeric
- Created total usage feature
- Created total call volume feature
- Created service pressure ratio